In [10]:
import dlib
import cv2
import numpy as np
import os

def get_upper_face_crops(image, detector, predictor,
                         forehead_padding=20,
                         eye_bottom_padding=20,
                         malar_padding=10):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)
    crops = []
    for face in faces:
        landmarks = predictor(gray, face)

        # Face bounding box coordinates
        face_top = max(0, face.top())
        face_bottom = min(gray.shape[0], face.bottom())

        # Eye and eyebrow landmarks
        left_eye = np.array([[landmarks.part(i).x, landmarks.part(i).y] for i in range(36, 42)])
        right_eye = np.array([[landmarks.part(i).x, landmarks.part(i).y] for i in range(42, 48)])
        left_eyebrow = np.array([[landmarks.part(i).x, landmarks.part(i).y] for i in range(17, 22)])
        right_eyebrow = np.array([[landmarks.part(i).x, landmarks.part(i).y] for i in range(22, 27)])

        # Forehead + eye bounding box
        x_min_eye = min(left_eye[:, 0].min(), right_eye[:, 0].min(),
                        left_eyebrow[:, 0].min(), right_eyebrow[:, 0].min())
        x_max_eye = max(left_eye[:, 0].max(), right_eye[:, 0].max(),
                        left_eyebrow[:, 0].max(), right_eyebrow[:, 0].max())

        y_eyebrows_top = min(left_eyebrow[:, 1].min(), right_eyebrow[:, 1].min())

        # Make sure crop top doesn't go above face bounding box
        y_min_eye = max(0, y_eyebrows_top - forehead_padding)
        y_min_eye = min(y_min_eye, face_top)

        y_eyes_bottom = max(left_eye[:, 1].max(), right_eye[:, 1].max())
        y_max_eye = min(gray.shape[0], y_eyes_bottom + eye_bottom_padding)

        roi_eye_forehead = image[y_min_eye:y_max_eye, x_min_eye:x_max_eye]  # Keep color crop as BGR

        # Only add if crop is valid size
        if roi_eye_forehead.size != 0 and roi_eye_forehead.shape[0] > 10 and roi_eye_forehead.shape[1] > 10:
            # Resize crop to fixed size (e.g., 128x64)
            crop_resized = cv2.resize(roi_eye_forehead, (128, 64), interpolation=cv2.INTER_AREA)
            crops.append(('eye_forehead', crop_resized))

    return crops

def save_dataset_cropped(dataset_path, output_path, detector, predictor, emotion_labels, target_count=1000):
    os.makedirs(output_path, exist_ok=True)

    for emotion in emotion_labels:
        emotion_input_folder = os.path.join(dataset_path, emotion)
        emotion_output_folder = os.path.join(output_path, emotion)
        os.makedirs(emotion_output_folder, exist_ok=True)

        if not os.path.exists(emotion_input_folder):
            print(f"Input folder for {emotion} not found, skipping.")
            continue

        saved_count = 0

        for imgfile in os.listdir(emotion_input_folder):
            if saved_count >= target_count:
                break  # Limit to target_count images per class
            img_path = os.path.join(emotion_input_folder, imgfile)
            img = cv2.imread(img_path)
            if img is None:
                print(f"Cannot load image {img_path}, skipping.")
                continue

            crops = get_upper_face_crops(img, detector, predictor)
            if not crops:
                print(f"No face detected or no valid crops from {img_path}, skipping.")
                continue

            for idx, (crop_name, crop) in enumerate(crops):
                if saved_count >= target_count:
                    break
                save_path = os.path.join(emotion_output_folder,
                                         f"{os.path.splitext(imgfile)[0]}_{crop_name}_crop{idx}.png")
                cv2.imwrite(save_path, crop)
                saved_count += 1

        print(f"Saved {saved_count} images for emotion '{emotion}'.")

    print("Dataset saved successfully.")

# Initialize dlib detector and predictor
emotion_labels = ['angry', 'fear', 'happy', 'sad']
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(r"F:\Academic\7th semester\FYP\recommondation_agents_implementation\shape_predictor_68_face_landmarks.dat")

# Paths to dataset and output directory
data_path = r"D:\our_dataset\rashmitha 4tos"
output_path = r"D:\FL_eye_dataset\New folder\rashmitha"

# Run crop and save
save_dataset_cropped(data_path, output_path, detector, predictor, emotion_labels, target_count=1000)


Saved 5 images for emotion 'angry'.
Saved 7 images for emotion 'fear'.
Saved 8 images for emotion 'happy'.
No face detected or no valid crops from D:\our_dataset\rashmitha 4tos\sad\Screenshot 2025-05-15 094648.png, skipping.
No face detected or no valid crops from D:\our_dataset\rashmitha 4tos\sad\Screenshot 2025-05-15 094701.png, skipping.
No face detected or no valid crops from D:\our_dataset\rashmitha 4tos\sad\Screenshot 2025-05-15 094712.png, skipping.
No face detected or no valid crops from D:\our_dataset\rashmitha 4tos\sad\Screenshot 2025-05-15 094728.png, skipping.
No face detected or no valid crops from D:\our_dataset\rashmitha 4tos\sad\Screenshot 2025-05-15 094740.png, skipping.
Saved 9 images for emotion 'sad'.
Dataset saved successfully.


In [3]:
import os
import cv2
import numpy as np
from glob import glob
from shutil import copy2
import random

# Constants
TARGET_WIDTH = 128      # Target width for resized images (larger RGB chosen size)
TARGET_HEIGHT = 64      # Target height
MIN_IMAGES_PER_CLASS = 1000

# Paths (example)
RGB_DATA_TRAIN = r"D:\eye_dataset\eye reaction last\train"
RGB_DATA_TEST = r"D:\eye_dataset\eye reaction last\test"
GRAYSCALE_DATA = r"D:\eye_dataset\Eye_cropped_6emotions"
OUTPUT_DIR = r"D:\eye_dataset\eye_dataset_1000_6classes"

CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise']


def load_images_from_folder(folder, size=(TARGET_WIDTH, TARGET_HEIGHT)):
    images = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is None:
            continue
         # Convert to grayscale
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_resized = cv2.resize(img_gray, size)
        images.append(img_resized)
    return images


def load_dataset_images(base_path, size=(TARGET_WIDTH, TARGET_HEIGHT)):
    dataset_images = {}
    for cls in CLASS_NAMES:
        cls_folder = os.path.join(base_path, cls)
        if not os.path.exists(cls_folder):
            dataset_images[cls] = []
            print(f"Warning: {cls_folder} does not exist")
            continue
        imgs = load_images_from_folder(cls_folder, size)
        dataset_images[cls] = imgs
    return dataset_images


def load_grayscale_images(base_path, size=(TARGET_WIDTH, TARGET_HEIGHT)):
    grayscale_images = {}
    for cls in CLASS_NAMES:
        cls_folder = os.path.join(base_path, cls)
        if not os.path.exists(cls_folder):
            grayscale_images[cls] = []
            print(f"Warning: {cls_folder} does not exist")
            continue
        imgs = []
        for filename in os.listdir(cls_folder):
            img_path = os.path.join(cls_folder, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img_resized = cv2.resize(img, size)
            # Convert grayscale to 3-channel by repeating channels
            img_3c = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2BGR)
            imgs.append(img_3c)
        grayscale_images[cls] = imgs
    return grayscale_images


def combine_datasets(rgb_train_path, rgb_test_path, grayscale_path, output_path):
    # Load RGB images from train and test combined
    rgb_train_images = load_dataset_images(rgb_train_path)
    rgb_test_images = load_dataset_images(rgb_test_path)
    grayscale_images = load_grayscale_images(grayscale_path)
    
    os.makedirs(output_path, exist_ok=True)
    
    for cls in CLASS_NAMES:
        rgb_imgs = rgb_train_images.get(cls, []) + rgb_test_images.get(cls, [])
        num_rgb = len(rgb_imgs)
        num_gray_needed = max(0, MIN_IMAGES_PER_CLASS - num_rgb)
        
        gray_imgs = grayscale_images.get(cls, [])
        random.shuffle(gray_imgs)  # randomize grayscale selection
        
        selected_gray = gray_imgs[:num_gray_needed]
        
        combined_imgs = rgb_imgs + selected_gray
        print(f"Class {cls}: RGB={num_rgb}, Grayscale added={len(selected_gray)}, Total={len(combined_imgs)}")
        
        # Save combined images
        class_output_folder = os.path.join(output_path, cls)
        os.makedirs(class_output_folder, exist_ok=True)
        
        for idx, img in enumerate(combined_imgs):
            save_path = os.path.join(class_output_folder, f"{cls}_{idx:03d}.png")
            cv2.imwrite(save_path, img)
            

# Example Usage

combine_datasets(
    rgb_train_path=RGB_DATA_TRAIN,
    rgb_test_path=RGB_DATA_TEST,
    grayscale_path=GRAYSCALE_DATA,
    output_path=OUTPUT_DIR
)


Class angry: RGB=72, Grayscale added=928, Total=1000
Class disgust: RGB=60, Grayscale added=940, Total=1000
Class fear: RGB=68, Grayscale added=932, Total=1000
Class happy: RGB=79, Grayscale added=921, Total=1000
Class sad: RGB=66, Grayscale added=934, Total=1000
Class surprise: RGB=63, Grayscale added=937, Total=1000


In [4]:
import os
import cv2
import random
from PIL import Image, ImageChops

# Constants
TARGET_WIDTH = 128       # Target width for resized images
TARGET_HEIGHT = 64       # Target height
MIN_IMAGES_PER_CLASS = 150

# Paths (example)
RGB_DATA_TRAIN = r"D:\eye_dataset\eye reaction last\train"
RGB_DATA_TEST = r"D:\eye_dataset\eye reaction last\test"
RGB_DATA = r"D:\eye_dataset\colored_eye_cropped"
OUTPUT_DIR = r"D:\eye_dataset\150_eye_RGB_new"

CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise']


def is_rgb_image(img_path):
    img = Image.open(img_path)
    
    # Convert to RGB mode if not already
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    r, g, b = img.split()
    
    # Check if any of the channels differ (not grayscale)
    if ImageChops.difference(r, g).getbbox() is None and ImageChops.difference(r, b).getbbox() is None:
        # All channels are same -> grayscale
        return False
    return True


def load_images_from_folder(folder, size=(TARGET_WIDTH, TARGET_HEIGHT)):
    images = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        
        # Check if image is RGB color (skip grayscale)
        if not is_rgb_image(img_path):
            continue
        
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        img_resized = cv2.resize(img, size)
        images.append(img_resized)
    return images


def load_dataset_images(base_path, size=(TARGET_WIDTH, TARGET_HEIGHT)):
    dataset_images = {}
    for cls in CLASS_NAMES:
        cls_folder = os.path.join(base_path, cls)
        if not os.path.exists(cls_folder):
            dataset_images[cls] = []
            print(f"Warning: {cls_folder} does not exist")
            continue
        imgs = load_images_from_folder(cls_folder, size)
        dataset_images[cls] = imgs
    return dataset_images


def combine_datasets(rgb_train_path, rgb_test_path, rgb_images_path, output_path):
    rgb_train_images = load_dataset_images(rgb_train_path)
    rgb_test_images = load_dataset_images(rgb_test_path)
    rgb_images = load_dataset_images(rgb_images_path)

    os.makedirs(output_path, exist_ok=True)

    for cls in CLASS_NAMES:
        combined = rgb_train_images.get(cls, []) + rgb_test_images.get(cls, []) + rgb_images.get(cls, [])
        num_combined = len(combined)
        print(f"Class {cls}: Total combined RGB images before selection = {num_combined}")

        # Shuffle combined images randomly to mix datasets well
        random.shuffle(combined)

        # Limit to first MIN_IMAGES_PER_CLASS images
        selected_imgs = combined[:MIN_IMAGES_PER_CLASS]

        if len(selected_imgs) < MIN_IMAGES_PER_CLASS:
            print(f"Warning: Class {cls} has only {len(selected_imgs)} RGB images, less than minimum {MIN_IMAGES_PER_CLASS}")

        class_output_folder = os.path.join(output_path, cls)
        os.makedirs(class_output_folder, exist_ok=True)

        for idx, img in enumerate(selected_imgs):
            save_path = os.path.join(class_output_folder, f"{cls}_{idx:03d}.png")
            cv2.imwrite(save_path, img)


# Example Usage
combine_datasets(
    rgb_train_path=RGB_DATA_TRAIN,
    rgb_test_path=RGB_DATA_TEST,
    rgb_images_path=RGB_DATA,
    output_path=OUTPUT_DIR
)


Class angry: Total combined RGB images before selection = 1014
Class disgust: Total combined RGB images before selection = 1018
Class fear: Total combined RGB images before selection = 1008
Class happy: Total combined RGB images before selection = 1047
Class sad: Total combined RGB images before selection = 1027
Class surprise: Total combined RGB images before selection = 1026


Dataset image Quality assesment

In [5]:
import os
import cv2
import numpy as np
from PIL import Image, ImageStat, ImageChops
import math


def is_grayscale(img_path):
    img = Image.open(img_path)
    # If mode is grayscale 'L'
    if img.mode == 'L':
        return True
    # Convert other modes to RGB
    if img.mode != 'RGB':
        img = img.convert('RGB')
    r, g, b = img.split()
    # Check if all channels are identical
    if ImageChops.difference(r, g).getbbox() is None and ImageChops.difference(r, b).getbbox() is None:
        return True
    else:
        return False


def get_num_channels(img_path):
    img = Image.open(img_path)
    return len(img.getbands())  # Returns number of channels


def analyze_image_quality(image_path):
    # Read image in grayscale for quality metrics
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Image size (height, width)
    size = image.shape

    # Sharpness (variance of the Laplacian)
    laplacian = cv2.Laplacian(image, cv2.CV_64F)
    sharpness = laplacian.var()

    # Contrast (standard deviation of pixel intensities)
    contrast = image.std()

    # Brightness (perceived brightness using PIL)
    img_pil = Image.open(image_path).convert('RGB')
    stat = ImageStat.Stat(img_pil)
    r, g, b = stat.mean
    brightness = math.sqrt(0.241 * (r ** 2) + 0.691 * (g ** 2) + 0.068 * (b ** 2))

    # Determine color type
    grayscale_flag = is_grayscale(image_path)
    color_type = "Grayscale" if grayscale_flag else "RGB"

    # Get number of channels
    channels = get_num_channels(image_path)

    return {
        "size": size,
        "sharpness": sharpness,
        "contrast": contrast,
        "brightness": brightness,
        "color_type": color_type,
        "channels": channels
    }


def analyze_dataset_images(folder_path):
    results = {}
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            image_path = os.path.join(folder_path, filename)
            info = analyze_image_quality(image_path)
            results[filename] = info
    return results


# Example usage:
dataset_folder = r"D:\eye_dataset\150_eye_RGB_new\angry"
image_info = analyze_dataset_images(dataset_folder)

for img_name, info in image_info.items():
    print(f"Image: {img_name}")
    print(f" Size (HxW): {info['size']}")
    print(f" Sharpness (Laplacian Variance): {info['sharpness']:.2f}")
    print(f" Contrast (Std Dev): {info['contrast']:.2f}")
    print(f" Brightness (Perceived): {info['brightness']:.2f}")
    print(f" Color Type: {info['color_type']}")
    print(f" Channels: {info['channels']}")
    print("-------------------------------")


Image: angry_000.png
 Size (HxW): (64, 128)
 Sharpness (Laplacian Variance): 1216.66
 Contrast (Std Dev): 71.64
 Brightness (Perceived): 167.23
 Color Type: RGB
 Channels: 3
-------------------------------
Image: angry_001.png
 Size (HxW): (64, 128)
 Sharpness (Laplacian Variance): 915.02
 Contrast (Std Dev): 34.15
 Brightness (Perceived): 81.30
 Color Type: RGB
 Channels: 3
-------------------------------
Image: angry_002.png
 Size (HxW): (64, 128)
 Sharpness (Laplacian Variance): 1162.03
 Contrast (Std Dev): 73.00
 Brightness (Perceived): 184.65
 Color Type: RGB
 Channels: 3
-------------------------------
Image: angry_003.png
 Size (HxW): (64, 128)
 Sharpness (Laplacian Variance): 79.73
 Contrast (Std Dev): 45.59
 Brightness (Perceived): 99.48
 Color Type: RGB
 Channels: 3
-------------------------------
Image: angry_004.png
 Size (HxW): (64, 128)
 Sharpness (Laplacian Variance): 149.99
 Contrast (Std Dev): 38.72
 Brightness (Perceived): 108.36
 Color Type: RGB
 Channels: 3
--------

Crop Eye reigon and create a Dataset

In [8]:
"""
EYE REGION EXTRACTOR FOR EMOTION RECOGNITION
============================================

Extracts eye, eyebrow, and forehead regions from videos
- 50 images per emotion
- No aspect ratio distortion
- High quality images

Input Structure:
video_dataset/
├── angry/
│   ├── video1.mp4
│   └── video2.mp4
├── fear/
├── happy/
└── sad/

Output Structure:
eye_dataset/
├── angry/    (50 eye region images)
├── fear/     (50 eye region images)
├── happy/    (50 eye region images)
└── sad/      (50 eye region images)
"""

import os
import cv2
import numpy as np
from tqdm import tqdm
import json

# ===================== CONFIGURATION =====================

# IMPORTANT: Update these paths
VIDEO_DATASET_PATH = r"F:\Academic\7th semester\FYP\Custom Video dataset\4_person\rashmitha"  # Your video folder
OUTPUT_DATASET_PATH = r"D:\FL_eye_dataset\rashmitha1"  # Where to save eye images

# Dataset structure
EMOTIONS = ['angry', 'fear', 'happy', 'sad']

# Extraction settings
IMAGES_PER_EMOTION = 50  # Exactly 50 images per emotion
TARGET_IMAGE_SIZE = 224  # Output will be 224x224

# Eye region settings
EYE_REGION_HEIGHT_RATIO = 0.6  # Extract top 60% of face (eyes + forehead)
EYE_REGION_EXPAND_W = 0.1  # 10% extra width on each side
INCLUDE_FOREHEAD_EXTRA = 0.3  # 30% extra above eyebrows for forehead

# Quality settings
JPEG_QUALITY = 95
MIN_FACE_SIZE = 80
BLUR_THRESHOLD = 10  # Lower = accept more images
MIN_FRAME_DISTANCE = 10  # Frames between extractions

# Video extensions
VIDEO_EXTENSIONS = ['.mp4', '.avi', '.mov', '.mkv', '.mpeg', '.flv', '.wmv']

# ===================== INITIALIZATION =====================

# Load face and eye detectors
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

print("✅ Loaded face and eye detectors")

# Statistics
stats = {
    'total_videos': 0,
    'total_frames_checked': 0,
    'total_images_saved': 0,
    'per_emotion': {}
}

# ===================== HELPER FUNCTIONS =====================

def is_blurry(image, threshold=BLUR_THRESHOLD):
    """Check if image is too blurry."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return variance < threshold


def detect_eye_region(frame):
    """
    Detect face and extract eye region (eyes, eyebrows, forehead).
    
    Returns:
        eye_region: Extracted eye region image
        success: True if successful extraction
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    
    # Detect face
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.05,
        minNeighbors=4,
        minSize=(MIN_FACE_SIZE, MIN_FACE_SIZE)
    )
    
    if len(faces) == 0:
        return None, False
    
    # Use largest face
    face = max(faces, key=lambda f: f[2] * f[3])
    x, y, w, h = face
    
    # Extract face region
    face_roi = frame[y:y+h, x:x+w]
    face_gray = gray[y:y+h, x:x+w]
    
    # Detect eyes in face region
    eyes = eye_cascade.detectMultiScale(
        face_gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(int(w * 0.1), int(h * 0.1))
    )
    
    # Method 1: If eyes detected, use them to define region
    if len(eyes) >= 2:
        # Sort eyes by x-coordinate (left to right)
        eyes_sorted = sorted(eyes, key=lambda e: e[0])
        
        # Get leftmost and rightmost eye
        left_eye = eyes_sorted[0]
        right_eye = eyes_sorted[-1]
        
        # Calculate eye region boundaries
        eye_x1 = left_eye[0]
        eye_x2 = right_eye[0] + right_eye[2]
        eye_y1 = min(left_eye[1], right_eye[1])
        eye_y2 = max(left_eye[1] + left_eye[3], right_eye[1] + right_eye[3])
        
        # Expand region to include eyebrows and forehead
        expand_w = int(w * EYE_REGION_EXPAND_W)
        expand_h_top = int(h * INCLUDE_FOREHEAD_EXTRA)
        expand_h_bottom = int((eye_y2 - eye_y1) * 0.2)  # Small expansion below eyes
        
        # Calculate final coordinates (relative to face)
        region_x1 = max(0, eye_x1 - expand_w)
        region_x2 = min(w, eye_x2 + expand_w)
        region_y1 = max(0, eye_y1 - expand_h_top)
        region_y2 = min(h, eye_y2 + expand_h_bottom)
        
        # Extract eye region
        eye_region = face_roi[region_y1:region_y2, region_x1:region_x2]
    
    # Method 2: If eyes not detected, use upper portion of face
    else:
        # Extract top 60% of face (covers eyes and forehead)
        region_height = int(h * EYE_REGION_HEIGHT_RATIO)
        eye_region = face_roi[0:region_height, :]
    
    if eye_region.size == 0:
        return None, False
    
    return eye_region, True


def resize_with_aspect_ratio(image, target_size):
    """
    Resize image to target_size x target_size maintaining aspect ratio.
    Adds black padding (letterboxing) if needed.
    """
    h, w = image.shape[:2]
    
    # Calculate scaling factor
    scale = target_size / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)
    
    # Resize image
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
    
    # Create black canvas
    canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    
    # Calculate position to center image
    y_offset = (target_size - new_h) // 2
    x_offset = (target_size - new_w) // 2
    
    # Place resized image on canvas
    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    
    return canvas


def extract_eye_regions_from_videos(video_paths, output_folder, target_count=50, emotion_name=""):
    """
    Extract exactly target_count eye region images from videos.
    
    Returns:
        (saved_count, frames_checked)
    """
    if not video_paths:
        return 0, 0
    
    print(f"      📹 Found {len(video_paths)} video(s)")
    
    candidates = []  # (video_path, frame_num, eye_region, quality_score)
    total_frames_checked = 0
    
    # Phase 1: Scan videos and collect candidate frames
    for video_idx, video_path in enumerate(video_paths):
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"      ⚠️ Could not open: {os.path.basename(video_path)}")
            continue
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        
        # Sample frames
        sample_interval = max(5, total_frames // (target_count * 5))
        
        frame_num = 0
        video_candidates = 0
        
        while frame_num < total_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ret, frame = cap.read()
            
            if not ret:
                break
            
            total_frames_checked += 1
            
            # Detect and extract eye region
            eye_region, success = detect_eye_region(frame)
            
            if success and eye_region is not None:
                # Check quality
                if not is_blurry(eye_region):
                    # Quality score based on size
                    quality_score = eye_region.shape[0] * eye_region.shape[1]
                    
                    candidates.append((video_path, frame_num, eye_region, quality_score))
                    video_candidates += 1
            
            frame_num += sample_interval
        
        cap.release()
        
        if video_candidates > 0:
            print(f"      ✓ Video {video_idx+1}: Found {video_candidates} valid eye regions")
    
    if not candidates:
        print(f"      ❌ No valid eye regions found")
        return 0, total_frames_checked
    
    print(f"      ✅ Found {len(candidates)} total eye region frames")
    
    # Phase 2: Select best diverse frames
    if len(candidates) <= target_count:
        selected = candidates
    else:
        # Sort by quality
        candidates.sort(key=lambda x: x[3], reverse=True)
        
        # Select frames ensuring diversity
        selected = []
        used_frames = []
        
        for video_path, frame_num, eye_region, quality in candidates:
            # Check if too close to already selected frames
            too_close = False
            for used_video, used_frame in used_frames:
                if used_video == video_path and abs(used_frame - frame_num) < MIN_FRAME_DISTANCE:
                    too_close = True
                    break
            
            if not too_close:
                selected.append((video_path, frame_num, eye_region, quality))
                used_frames.append((video_path, frame_num))
            
            if len(selected) >= target_count:
                break
    
    # Phase 3: Save selected eye regions
    print(f"      💾 Extracting {len(selected)} eye region images...")
    
    saved_count = 0
    
    for video_path, frame_num, eye_region, _ in selected:
        # Resize maintaining aspect ratio
        eye_region_resized = resize_with_aspect_ratio(eye_region, TARGET_IMAGE_SIZE)
        
        # Save image
        img_filename = f"eye_{saved_count+1:04d}.jpg"
        img_path = os.path.join(output_folder, img_filename)
        
        cv2.imwrite(img_path, eye_region_resized, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        saved_count += 1
    
    return saved_count, total_frames_checked


def find_videos_in_folder(folder_path):
    """Find all video files in a folder."""
    if not os.path.exists(folder_path):
        return []
    
    videos = []
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        if os.path.isfile(file_path):
            _, ext = os.path.splitext(file)
            if ext.lower() in VIDEO_EXTENSIONS:
                videos.append(file_path)
    
    return videos


def create_output_structure():
    """Create output folder structure."""
    print("\n📁 Creating output folder structure...")
    
    os.makedirs(OUTPUT_DATASET_PATH, exist_ok=True)
    
    for emotion in EMOTIONS:
        emotion_path = os.path.join(OUTPUT_DATASET_PATH, emotion)
        os.makedirs(emotion_path, exist_ok=True)
    
    print("   ✅ Folder structure created")


def process_dataset():
    """Main processing function."""
    print("="*80)
    print(" "*25 + "EYE REGION EXTRACTOR")
    print("="*80)
    
    # Validate paths
    if not os.path.exists(VIDEO_DATASET_PATH):
        print(f"\n❌ ERROR: Video dataset path not found!")
        print(f"   Path: {VIDEO_DATASET_PATH}")
        print(f"\n   Please update VIDEO_DATASET_PATH in the script.")
        return
    
    # Create output structure
    create_output_structure()
    
    # Display configuration
    print(f"\n⚙️ Configuration:")
    print(f"   Input:  {VIDEO_DATASET_PATH}")
    print(f"   Output: {OUTPUT_DATASET_PATH}")
    print(f"   Emotions: {len(EMOTIONS)} ({', '.join(EMOTIONS)})")
    print(f"   Target: {IMAGES_PER_EMOTION} eye region images per emotion")
    print(f"   Total Expected: {len(EMOTIONS) * IMAGES_PER_EMOTION} images")
    print(f"   Image Size: {TARGET_IMAGE_SIZE}x{TARGET_IMAGE_SIZE}")
    print(f"   Region: Eyes, eyebrows, forehead")
    
    print("\n" + "="*80)
    print("PROCESSING VIDEOS...")
    print("="*80 + "\n")
    
    # Process each emotion
    for emotion in EMOTIONS:
        print(f"\n🎭 Processing {emotion.upper()}")
        print("-" * 80)
        
        emotion_input_path = os.path.join(VIDEO_DATASET_PATH, emotion)
        emotion_output_path = os.path.join(OUTPUT_DATASET_PATH, emotion)
        
        if not os.path.exists(emotion_input_path):
            print(f"   ⚠️ Emotion folder not found: {emotion_input_path}")
            stats['per_emotion'][emotion] = 0
            continue
        
        # Find videos
        videos = find_videos_in_folder(emotion_input_path)
        
        if not videos:
            print(f"   ⚠️ No videos found")
            stats['per_emotion'][emotion] = 0
            continue
        
        # Extract eye regions
        saved, checked = extract_eye_regions_from_videos(
            videos,
            emotion_output_path,
            IMAGES_PER_EMOTION,
            emotion
        )
        
        stats['per_emotion'][emotion] = saved
        stats['total_images_saved'] += saved
        stats['total_videos'] += len(videos)
        stats['total_frames_checked'] += checked
        
        # Status
        if saved == IMAGES_PER_EMOTION:
            print(f"   ✅ SUCCESS: {saved}/{IMAGES_PER_EMOTION} eye region images extracted")
        elif saved > 0:
            print(f"   ⚠️ PARTIAL: {saved}/{IMAGES_PER_EMOTION} eye region images extracted")
            print(f"   💡 Tip: Add more videos or lower quality thresholds")
        else:
            print(f"   ❌ FAILED: No images extracted")
            print(f"   💡 Tip: Check video quality, lighting, and face visibility")
    
    # Final summary
    print("\n" + "="*80)
    print("PROCESSING COMPLETE!")
    print("="*80)
    
    total_expected = len(EMOTIONS) * IMAGES_PER_EMOTION
    
    print(f"\n📊 FINAL STATISTICS:")
    print(f"   Total videos processed: {stats['total_videos']}")
    print(f"   Total frames checked: {stats['total_frames_checked']}")
    print(f"   Total eye images extracted: {stats['total_images_saved']}/{total_expected}")
    print(f"   Success rate: {stats['total_images_saved']/total_expected*100:.1f}%")
    
    print(f"\n🎭 Per Emotion Breakdown:")
    for emotion, count in stats['per_emotion'].items():
        status = "✅" if count == IMAGES_PER_EMOTION else "⚠️" if count > 0 else "❌"
        print(f"   {status} {emotion:10s}: {count:2d}/{IMAGES_PER_EMOTION} images")
    
    # Save statistics
    stats_file = os.path.join(OUTPUT_DATASET_PATH, 'extraction_stats.json')
    with open(stats_file, 'w') as f:
        json.dump(stats, f, indent=2)
    
    print(f"\n✅ Statistics saved to: {stats_file}")
    print(f"📂 Eye region dataset: {OUTPUT_DATASET_PATH}")


def verify_dataset():
    """Verify the created dataset."""
    print("\n" + "="*80)
    print("DATASET VERIFICATION")
    print("="*80 + "\n")
    
    if not os.path.exists(OUTPUT_DATASET_PATH):
        print("❌ Output dataset not found!")
        return
    
    total_images = 0
    
    for emotion in EMOTIONS:
        emotion_path = os.path.join(OUTPUT_DATASET_PATH, emotion)
        
        if os.path.exists(emotion_path):
            images = [f for f in os.listdir(emotion_path) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            count = len(images)
            total_images += count
            
            status = "✅" if count == IMAGES_PER_EMOTION else "⚠️" if count > 0 else "❌"
            print(f"{status} {emotion:10s}: {count:3d} images")
        else:
            print(f"❌ {emotion:10s}: Missing folder")
    
    print(f"\n📊 Total: {total_images} images")
    expected = len(EMOTIONS) * IMAGES_PER_EMOTION
    print(f"   Expected: {expected} images")
    print(f"   Completeness: {total_images/expected*100:.1f}%")


# ===================== MAIN EXECUTION =====================

if __name__ == "__main__":
    print("\n" + "="*80)
    
    try:
        # Process dataset
        process_dataset()
        
        # Verify results
        verify_dataset()
        
        print("\n" + "="*80)
        print("✅ ALL DONE!")
        print("="*80)
        print("\n💡 Your eye region dataset is ready!")
        print(f"   Location: {OUTPUT_DATASET_PATH}")
        print(f"   Format: {TARGET_IMAGE_SIZE}x{TARGET_IMAGE_SIZE} images")
        print(f"   Content: Eyes, eyebrows, forehead regions")
        
    except KeyboardInterrupt:
        print("\n\n⚠️ Process interrupted by user")
    except Exception as e:
        print(f"\n\n❌ Error occurred: {e}")
        import traceback
        traceback.print_exc()

✅ Loaded face and eye detectors

                         EYE REGION EXTRACTOR

📁 Creating output folder structure...
   ✅ Folder structure created

⚙️ Configuration:
   Input:  F:\Academic\7th semester\FYP\Custom Video dataset\4_person\rashmitha
   Output: D:\FL_eye_dataset\rashmitha1
   Emotions: 4 (angry, fear, happy, sad)
   Target: 50 eye region images per emotion
   Total Expected: 200 images
   Image Size: 224x224
   Region: Eyes, eyebrows, forehead

PROCESSING VIDEOS...


🎭 Processing ANGRY
--------------------------------------------------------------------------------
      📹 Found 9 video(s)
      ✓ Video 1: Found 20 valid eye regions
      ✓ Video 2: Found 11 valid eye regions
      ✓ Video 3: Found 16 valid eye regions
      ✓ Video 4: Found 16 valid eye regions
      ✓ Video 5: Found 24 valid eye regions
      ✓ Video 6: Found 20 valid eye regions
      ✓ Video 7: Found 16 valid eye regions
      ✓ Video 8: Found 20 valid eye regions
      ✓ Video 9: Found 16 valid eye re